**Eye-Tracking & Neurocognitive Intelligence | Eye-Movement Modelling & Advanced AI | Presented by: Reza Saadatyar (2026)**

**02 | Prospective 100-person Main Study (not yet executed)**

![Original GazeBase feature inputs, four classical model results and independent-person Validation](../../images/02_statistical_and_classical_ml.svg)

This notebook is ready for a new **100-person** main-cohort run after Notebook 01. The participant count is set in the single configuration cell of Notebook 01 and can be changed to 300 for a separately specified run. The image and earlier Git history document the completed **ten-person pilot**; they are not results of the new cohort.

For a fresh Colab Runtime, upload the matching Notebook 01 ZIP into /content. Explicitly uploaded stage ZIPs take priority over stale pilot files present in the checkout. The new main-cohort protocol and independent-person split are checked before analysis. No Google Drive is used. No new main-cohort Test metrics have been observed.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_movement_modelling")

# ---------------------------- 2. Data and Models -----------------------------
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from IPython.display import display

# ------------------------ 3. Research Infrastructure -------------------------
from s2_handoff import ensure_s2_stage_inputs, verify_main_cohort
from core.output_download import download_outputs_from_colab
from core.plot_style import plot_style, save_figure
from s2_data import TASKS, CLASS_TO_INDEX, load_protocol
from s2_classical import classical_candidates
from s2_metrics import (
    FEATURE_COLUMN_NAMES, require_split_consistency,
    classical_record_predictions, metric_row,
)

# ----------------------------- 4. Output Context -----------------------------
output_dir = root / "eye_movement_modelling" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(font_family="DejaVu Sans", figure_dpi=100)
print("Current study outputs:", output_dir)

**02. Source Feature Inputs**

Read the current stage-01 features, person partitions and protocol. The study-specific ZIP restore helper uses an uploaded new-cohort ZIP instead of stale checkout pilot outputs. Verify source identities and participant counts before fitting. This section is pending execution on the chosen main cohort. Linked figures remain historical pilot illustrations until the new run produces updated numerical figures.


In [ ]:
# ======================== 02. Recover Record Features ========================

# ----------------------- 1. Named Current Input Files ------------------------
ensure_s2_stage_inputs(
    output_dir, 1,
    ("01_record_features.csv", "01_split_assignments.csv",
     "01_protocol.csv"),
)
features = pd.read_csv(
    output_dir / "01_record_features.csv",
    dtype={"participant_id": str, "record_id": str},
)
split = pd.read_csv(
    output_dir / "01_split_assignments.csv",
    dtype={"participant_id": str, "record_id": str},
)
protocol = load_protocol(output_dir / "01_protocol.csv")
require_split_consistency(features, split)
role_counts = verify_main_cohort(protocol, split)
print("Main cohort participant roles:", role_counts)

# --------------------- 2. Separate Train and Validation ----------------------
train = features.loc[features.split.eq("train")].copy()
validation = features.loc[features.split.eq("validation")].copy()
feature_names = list(FEATURE_COLUMN_NAMES)
if train.empty or validation.empty:
    raise ValueError("No Train/Validation task records were selected")
print("Protocol purpose:", protocol["purpose"])
print("Train records / participants:", len(train),
      train.participant_id.nunique())
print("Validation records / participants:", len(validation),
      validation.participant_id.nunique())
print("Input feature columns:", feature_names)
display(train[["record_id", "task", "valid_fraction"]].head(8))
candidates = classical_candidates(seed=protocol["split_seed"])
reported = []


**03. Class-Prior Baseline**

Fit the training-derived prior-only classifier on the new Train recordings. Evaluate records from the independent Validation people, never Test. This section is pending execution on the chosen main cohort. Linked figures remain historical pilot illustrations until the new run produces updated numerical figures.


In [ ]:
# ==================== 03. No-Signal Class-Prior Baseline =====================

# --------------------------- 1. Fit Train Records ----------------------------
name = "class_prior"
model = candidates[name]
model.fit(train[feature_names], train["task"].map(CLASS_TO_INDEX))
joblib.dump(model, output_dir / f"02_{name}.joblib")

# ---------------------- 2. Evaluate Held-Out Validation ----------------------
pred = classical_record_predictions(
    model, validation, model_name=name, partition="validation",
)
pred.to_csv(output_dir / f"02_{name}_validation_predictions.csv", index=False)
row = metric_row(
    pred.target_index, pred.pred_index, model=name,
    partition="validation",
    n_participants=validation["participant_id"].nunique(),
)
reported.append(row)
display(pd.DataFrame([row]))

# ---------------------- 3. Display Model Error Pattern -----------------------
matrix = confusion_matrix(
    pred.target_index, pred.pred_index, labels=range(len(TASKS)),
)
fig, ax = plt.subplots(figsize=(5.6, 4.8))
ConfusionMatrixDisplay(matrix, display_labels=TASKS).plot(
    ax=ax, colorbar=False, values_format="d",
)
ax.set_title("Class Prior: Validation by Record")
fig.tight_layout()
save_figure(fig, output_dir / f"02_{name}_confusion.png")
plt.show()


**04. Logistic Regression**

Fit the unchanged, class-weighted Logistic Regression pipeline with Train-only imputation and standardisation. Save current-cohort Validation predictions and the confusion matrix. This section is pending execution on the chosen main cohort. Linked figures remain historical pilot illustrations until the new run produces updated numerical figures.


In [ ]:
# ======================= 04. Logistic Regression Model =======================

# --------------------------- 1. Fit Train Records ----------------------------
name = "logistic_regression"
model = candidates[name]
model.fit(train[feature_names], train["task"].map(CLASS_TO_INDEX))
joblib.dump(model, output_dir / f"02_{name}.joblib")

# ---------------------- 2. Evaluate Held-Out Validation ----------------------
pred = classical_record_predictions(
    model, validation, model_name=name, partition="validation",
)
pred.to_csv(output_dir / f"02_{name}_validation_predictions.csv", index=False)
row = metric_row(
    pred.target_index, pred.pred_index, model=name,
    partition="validation",
    n_participants=validation["participant_id"].nunique(),
)
reported.append(row)
display(pd.DataFrame([row]))

# ---------------------- 3. Display Model Error Pattern -----------------------
matrix = confusion_matrix(
    pred.target_index, pred.pred_index, labels=range(len(TASKS)),
)
fig, ax = plt.subplots(figsize=(5.6, 4.8))
ConfusionMatrixDisplay(matrix, display_labels=TASKS).plot(
    ax=ax, colorbar=False, values_format="d",
)
ax.set_title("Logistic Regression: Validation by Record")
fig.tight_layout()
save_figure(fig, output_dir / f"02_{name}_confusion.png")
plt.show()


**05. Random Forest**

Fit the same fixed Random Forest configuration on Train only, and evaluate on the same new Validation people as the other candidates. This section is pending execution on the chosen main cohort. Linked figures remain historical pilot illustrations until the new run produces updated numerical figures.


In [ ]:
# ========================== 05. Random Forest Model ==========================

# --------------------------- 1. Fit Train Records ----------------------------
name = "random_forest"
model = candidates[name]
model.fit(train[feature_names], train["task"].map(CLASS_TO_INDEX))
joblib.dump(model, output_dir / f"02_{name}.joblib")

# ---------------------- 2. Evaluate Held-Out Validation ----------------------
pred = classical_record_predictions(
    model, validation, model_name=name, partition="validation",
)
pred.to_csv(output_dir / f"02_{name}_validation_predictions.csv", index=False)
row = metric_row(
    pred.target_index, pred.pred_index, model=name,
    partition="validation",
    n_participants=validation["participant_id"].nunique(),
)
reported.append(row)
display(pd.DataFrame([row]))

# ---------------------- 3. Display Model Error Pattern -----------------------
matrix = confusion_matrix(
    pred.target_index, pred.pred_index, labels=range(len(TASKS)),
)
fig, ax = plt.subplots(figsize=(5.6, 4.8))
ConfusionMatrixDisplay(matrix, display_labels=TASKS).plot(
    ax=ax, colorbar=False, values_format="d",
)
ax.set_title("Random Forest: Validation by Record")
fig.tight_layout()
save_figure(fig, output_dir / f"02_{name}_confusion.png")
plt.show()


**06. Gradient Boosting**

Fit the fixed gradient-boosting pipeline with Train-only preprocessing. Its parameters are unchanged; the cohort is different. This section is pending execution on the chosen main cohort. Linked figures remain historical pilot illustrations until the new run produces updated numerical figures.


In [ ]:
# ======================== 06. Gradient Boosting Model ========================

# --------------------------- 1. Fit Train Records ----------------------------
name = "gradient_boosting"
model = candidates[name]
model.fit(train[feature_names], train["task"].map(CLASS_TO_INDEX))
joblib.dump(model, output_dir / f"02_{name}.joblib")

# ---------------------- 2. Evaluate Held-Out Validation ----------------------
pred = classical_record_predictions(
    model, validation, model_name=name, partition="validation",
)
pred.to_csv(output_dir / f"02_{name}_validation_predictions.csv", index=False)
row = metric_row(
    pred.target_index, pred.pred_index, model=name,
    partition="validation",
    n_participants=validation["participant_id"].nunique(),
)
reported.append(row)
display(pd.DataFrame([row]))

# ---------------------- 3. Display Model Error Pattern -----------------------
matrix = confusion_matrix(
    pred.target_index, pred.pred_index, labels=range(len(TASKS)),
)
fig, ax = plt.subplots(figsize=(5.6, 4.8))
ConfusionMatrixDisplay(matrix, display_labels=TASKS).plot(
    ax=ax, colorbar=False, values_format="d",
)
ax.set_title("Gradient Boosting: Validation by Record")
fig.tight_layout()
save_figure(fig, output_dir / f"02_{name}_confusion.png")
plt.show()


**07. Validation Model Comparison**

Save the four current-cohort record-level metrics and comparison figure. Previously perfect two-person pilot scores are historical evidence, not predictions for this new experiment. This section is pending execution on the chosen main cohort. Linked figures remain historical pilot illustrations until the new run produces updated numerical figures.


In [ ]:
# ===================== 07. Compare Classical Validation ======================

# ----------------------- 1. Persist Comparable Scores ------------------------
metrics = pd.DataFrame(reported).sort_values(
    ["macro_f1", "balanced_accuracy", "model"],
    ascending=[False, False, True],
)
metrics.to_csv(output_dir / "02_validation_metrics.csv", index=False)
display(metrics)

# ------------------------ 2. Plot Record-Level Scores ------------------------
fig, ax = plt.subplots(figsize=(8.7, 4.7))
positions = np.arange(len(metrics))
ax.bar(positions - 0.17, metrics["macro_f1"], width=0.34,
       label="Macro F1")
ax.bar(positions + 0.17, metrics["balanced_accuracy"], width=0.34,
       label="Balanced accuracy")
ax.set_xticks(positions, metrics["model"], rotation=20, ha="right")
ax.set(ylabel="Validation score by recording",
       ylim=(0, 1), title="Classical Task Baselines: Validation Only")
ax.grid(axis="y", alpha=0.2)
ax.legend()
fig.tight_layout()
save_figure(fig, output_dir / "02_classical_comparison.png")
plt.show()


**08. Interpretation and Stage ZIP**

Package all stage-02 checkpoints, Validation predictions and figures into one current ZIP. Preserve it for Notebook 05. The main Test remains reserved. This section is pending execution on the chosen main cohort. Linked figures remain historical pilot illustrations until the new run produces updated numerical figures.


In [ ]:
# ==================== 08. Export Classical Model Evidence ====================
download_outputs_from_colab("eye_movement_modelling/output", "02")
